# 4 NEON Lidar Products and Metadata Shapefiles

This notebook will show how to download tile boundary shapefiles, which can be used to see the extent of each AOP L3 tile in the SOAP saddle site.

In addition, we will take a quick look at the lidar-derived models (CHM, DSM, DTM) for the tiles we are interested in looking at. 

For the CWC comparison between burned and un-burned areas, we would like to make sure that the two areas we are looking at are similar in most aspects, aside from whether or not the forest had been affected by the fire. As a first pass, we can look at histograms of the Canopy Height and Digital Terrain models to make sure we're comparing apples to apples, as closely as possible.

## 1. Setup

### 1.1 Import the required Python libraries.

In [ ]:
# Import required packages
import dotenv
import geopandas as gpd
import neonutilities as nu
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import os
import pandas as pd
import rasterio as rio
from rasterio.plot import show, show_hist
import sys
from zipfile import ZipFile

If not already installed, install the `neonutilities` and `python-dotenv` packages using `pip` as follows:
- `!pip install neonutilities`
- `!pip install python-dotenv`

### 1.2 NEON Lidar Data Download

In this section we will download all CHM tiles for the SOAP saddle site in both 2023 and 2024. As part of these data downloads, the associated Metadata folder contains shapefile boundaries. Depending on which year was downloaded, these may be zipped (year of acquisition > 2023) or unzipped (year of aquisition < 2023), so we can write a function that will handle both cases so that we can properly extract the data.

In [ ]:
# download all CHM tiles for the SOAP site in 2023
nu.by_file_aop(dpid='DP3.30015.001', # CHM
               site='SOAP',
               year='2023',
               token=os.environ.get("NEON_TOKEN"),
               )

In [ ]:
# download all CHM tiles for the SOAP site in 2024
# 2024 data is still provisional as of 2025, so you need to set include_provisional=True
nu.by_file_aop(dpid='DP3.30015.001', # CHM
               site='SOAP',
               year='2023',
               include_provisional=False,
               token=os.environ.get("NEON_TOKEN"),
               savepath='../../../data/SOAP/NEON')

In [ ]:
def list_files_in_subdirectories(start_directory):
    """
    Lists all files within a specified directory and its subdirectories.

    Args:
        start_directory (str): The path to the directory to start the search from.

    Returns:
        list: A list of full paths to all files found.
    """
    all_files = []
    for root, _, files in os.walk(start_directory):
        for file in files:
            full_path = os.path.join(root, file)
            all_files.append(full_path)
    return all_files

In [ ]:
# optionally show all files in the data\SOAP\NEON folder, this will be a long list if you run
list_files_in_subdirectories(r'..\..\..\data\SOAP\NEON')

In [ ]:
def find_data_subfolders(root_dir):
    """
    Recursively finds subfolders within a directory that contain data (files)
    and excludes subfolders that only contain other subfolders.

    Args:
        root_dir: The path to the root directory to search.

    Returns:
        A list of paths to the subfolders containing data.
    """
    data_subfolders = []
    for root, dirs, files in os.walk(root_dir):
        # Check if the current directory has both subdirectories and files
        if dirs and files:
            # Iterate through subdirectories to find those that contain files
            for dir_name in dirs:
                dir_path = os.path.join(root, dir_name)
                if any(os.path.isfile(os.path.join(dir_path, f)) for f in os.listdir(dir_path)):
                    data_subfolders.append(dir_path)
        # If the current directory has no subdirectories, but has files, we still want to keep the directory.
        elif files:
            if root != root_dir:  # Avoid adding the root directory itself if it has files
                data_subfolders.append(root)

    return data_subfolders

In [ ]:
subfolders = find_data_subfolders(r'..\..\..\data\SOAP\NEON')
subfolders = [s.replace('..\\..\\..\\data\\SOAP\\NEON\\','') for s in subfolders] # make the output a little tidier
subfolders #d isplay the subfolders

Here we can see that we've downloaded 2 data products, DP3.30006.002, and DP3.30015.001 (the CHM). We have downloaded the CHM data for 2 years. Data from 2023_SOAP_6 and 2023_SOAP_7 are stored in the neon-aop-products bucket, and data from 2024_SOAP_8 are saved in the `neon-aop-provisional-products` bucket, as those data are currently only available provisionally (i.e. they have not been included in RELEASE-2025). We can also see that there are files in the `L3\DiscreteLidar\CanopyHeightModelGtif` folder as well as files in a `Metadata` folder, including `TileBoundary` files. These files store both kml and shapefiles (.shp) containing the boundaries of the individual tiles, as well as the entire site. Let's take a look at the Metadata\DiscreteLidar\TileBoundary\shps folder from `2023_SOAP_7` to start:

In [ ]:
soap_shps = os.listdir(os.path.join('..\\..\\..\\data\\SOAP\\NEON',r'DP3.30015.001\\neon-aop-products\\2023\\FullSite\\D17\\2023_SOAP_7\\Metadata\\DiscreteLidar\\TileBoundary\\shps'))
print(f"Files found in {r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\DP3.30015.001\neon-aop-products\2023\FullSite\D17\2023_SOAP_7\Metadata\DiscreteLidar\TileBoundary\shps\NEON_D17_SOAP_DPQA_292000_4095000_boundary.shp"}: {soap_shps}")

In [ ]:
# get the full path of the shape files, only include the .shp extension
# the .shx extension must be in the same folder as the .shp files
soap_shps_fullpath = [r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\DP3.30015.001\neon-aop-products\2023\FullSite\D17\2023_SOAP_7\Metadata\DiscreteLidar\TileBoundary\shps\NEON_D17_SOAP_DPQA_292000_4095000_boundary.shp" + f for f in soap_shps if f.endswith('.shp')]
# soap_shps_fullpath #optionally display the shapefiles

The shps folder contains shapefiles for each of the lidar data tiles. The UTM x, y position of the SW (lower-left) corner is provided in the file name, similar to the data products. These tile boundaries will also be the same for the non-lidar products (reflectance and camera). There are 4 different file types: .dbf, .prj, .shp, .shx for each tile. These are all associated with the .shp file. We can use geopandas

In [ ]:
# Define the base directory where your shapefiles are located
# This should be the path to the folder containing the .shp files, NOT including the filename itself.
base_dir = r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\notebooks\exploratory\rn\DP3.30015.001\neon-aop-products\2023\FullSite\D17\2023_SOAP_7\Metadata\DiscreteLidar\TileBoundary\shps"

# Assume 'soap_shps' is a list of just the filenames (e.g., ['NEON_D17_SOAP_DPQA_292000_4095000_boundary.shp'])
# If you obtained 'soap_shps' by listing files in 'base_dir', this is correct.
# For example, to simulate:
# soap_shps = ['NEON_D17_SOAP_DPQA_292000_4095000_boundary.shp', 'another_tile.shp'] # Replace with your actual list

# Correctly construct the full paths using os.path.join
soap_shps_fullpath = [os.path.join(base_dir, f) for f in soap_shps if f.endswith('.shp')]

# Now, when you iterate, shp_path will be correct
all_soap_gdfs = []
for shp_path in soap_shps_fullpath:
     print(f"Attempting to read: {shp_path}") # Add this print statement for verification!
     gdf = gpd.read_file(shp_path)
     all_soap_gdfs.append(gdf)

# # Concatenate into a single GeoDataFrame
soap_tiles_gdf = gpd.GeoDataFrame(pd.concat(all_soap_gdfs, ignore_index=True))
soap_tiles_gdf.crs

In [ ]:
# if you get the warning message "Failed to auto identify EPSG: 7" you can set it manually as follows:
soap_tiles_gdf.set_crs("EPSG:32611")
fig, ax = plt.subplots(figsize=(10, 10))
soap_tiles_gdf.plot(ax=ax, edgecolor='black', alpha=0.5)
plt.title("2023_SOAP_7 Tile Boundaries")
plt.ticklabel_format(style='plain', axis='y') 
plt.show()

In [ ]:
# Define the absolute path to the creek fire shapefile
creek_fire_shp = r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\data\shapefiles\creek_fire\creek_fire_perimeter.shp"

try:
    creek_fire_gdf = gpd.read_file(creek_fire_shp)
    print('CRS of Creek Fire Boundary:', creek_fire_gdf.crs)

    # Ensure the CRS transformation is valid
    if creek_fire_gdf.crs != "EPSG:32611":
        creek_fire_gdf = creek_fire_gdf.to_crs("EPSG:32611") # Assign back if not inplace=True
        print("Creek Fire Boundary CRS transformed to EPSG:32611")
    else:
        print("Creek Fire Boundary already in EPSG:32611")

    fig, ax = plt.subplots(figsize=(10, 10))
    soap_tiles_gdf.plot(ax=ax, edgecolor='black', alpha=0.5)
    creek_fire_gdf.plot(ax=ax, color='orange', edgecolor='red', alpha=0.4)
    plt.title("Creek Fire Boundary Overlain with SOAP Tiles") # More descriptive title
    plt.ticklabel_format(style='plain', axis='y')
    plt.show()

except FileNotFoundError:
    print(f"Error: Creek Fire shapefile not found at: {creek_fire_shp}")
    print("Please verify the absolute path to 'creek_fire_perimeter.shp'.")
except Exception as e:
    print(f"An error occurred while processing the Creek Fire shapefile: {e}")

In [ ]:
# now let's show the creek fire boundary overlain with these tiles:
creek_fire_shp = r"C:\Users\stem2\Documents\Capstone\AOP-EMIT\data\shapefiles\creek_fire\creek_fire_perimeter.shp"
creek_fire_gdf = gpd.read_file(creek_fire_shp)
print('crs of creek fire boundary:', creek_fire_gdf.crs)
creek_fire_gdf.to_crs("EPSG:32611", inplace=True) #

fig, ax = plt.subplots(figsize=(10, 10))
soap_tiles_gdf.plot(ax=ax, edgecolor='black', alpha=0.5)
creek_fire_gdf.plot(ax=ax, color='orange', edgecolor='red', alpha=0.4)
plt.title("Creek Fire Boundary")
plt.ticklabel_format(style='plain', axis='y') 
plt.show()

In [ ]:
creek_fire = creek_fire_gdf.plot(color='red', figsize=(5,5))
soap_tiles_gdf.plot(ax=creek_fire, color='yellow', edgecolor='orange', alpha = 0.4)
plt.xlabel("UTM x", labelpad=12)
plt.ylabel("UTM y", labelpad=12)
plt.ticklabel_format(style='plain', axis='y') 
plt.xticks(rotation=30)

## Add legend
legend_elements = [Patch(facecolor='red', edgecolor='red', label='2020 Creek Fire perimeter'), 
                  Patch(facecolor='none', edgecolor='yellow', label='2023_SOAP_7 boundary')]
plt.legend(handles=legend_elements, bbox_to_anchor=(2.6,1))
plt.show()

Zoom in on the SOAP site so you can see the individual tiles a little better:

In [ ]:
creek_fire = creek_fire_gdf.plot(color='red', figsize=(5,5))
soap_tiles_gdf.plot(ax=creek_fire, color='yellow', edgecolor='orange', alpha = 0.4)
plt.xlabel("UTM x", labelpad=12)
plt.ylabel("UTM y", labelpad=12)
plt.ticklabel_format(style='plain', axis='y') 
plt.xticks(rotation=30)

## Add legend
legend_elements = [Patch(facecolor='red', edgecolor='red', label='2020 Creek Fire perimeter'), 
                  Patch(facecolor='none', edgecolor='yellow', label='2023_SOAP_7 boundary')]
plt.legend(handles=legend_elements, bbox_to_anchor=(2.6,1))

# Set x and y limits to zoom into a specific area
ax = plt.gca()
ax.set_xlim(290000, 303000)
ax.set_ylim(4095000, 4110000)

plt.show()

We want to compare an area that has been affected by the creek fire boundary with an area that was unaffected. To start, let's look at two adjacent tiles where one is within the creek fire boundary and one is outside. 

Initially, we can look at 298000_4100000 (burned) and 298000_4101000 (unburned). We will highlight these two tiles below:

In [ ]:
burned_tile_shp = '..\\..\\..\\data\\SOAP\\NEON\\DP3.30015.001\\neon-aop-products\\2023\\FullSite\\D17\\2023_SOAP_7\\Metadata\\DiscreteLidar\\TileBoundary\\shps\\NEON_D17_SOAP_DPQA_298000_4100000_boundary.shp'
unburned_tile_shp = '..\\..\\..\\data\\SOAP\\NEON\\DP3.30015.001\\neon-aop-products\\2023\\FullSite\\D17\\2023_SOAP_7\\Metadata\\DiscreteLidar\\TileBoundary\\shps\\NEON_D17_SOAP_DPQA_298000_4101000_boundary.shp'

burned_tile_gdf = gpd.read_file(burned_tile_shp)
unburned_tile_gdf = gpd.read_file(unburned_tile_shp)

In [ ]:
creek_fire = creek_fire_gdf.plot(color='red', figsize=(5,5))
soap_tiles_gdf.plot(ax=creek_fire, color='yellow', edgecolor='orange', alpha = 0.4)
burned_tile_gdf.plot(ax=creek_fire, color='grey', alpha=0.5)
unburned_tile_gdf.plot(ax=creek_fire, color='green', alpha=0.5)
plt.xlabel("UTM x", labelpad=12)
plt.ylabel("UTM y", labelpad=12)
plt.ticklabel_format(style='plain', axis='y') 
plt.xticks(rotation=30)

## Add legend
legend_elements = [Patch(facecolor='red', edgecolor='red', label='2020 Creek Fire perimeter'), 
                  Patch(facecolor='none', edgecolor='yellow', label='2023_SOAP_7 boundary'),
                  Patch(facecolor='grey', edgecolor='grey', label='burned tile'),
                  Patch(facecolor='green', edgecolor='green', label='unburned tile')]
plt.legend(handles=legend_elements, bbox_to_anchor=(2.6,1))

# Set x and y limits to zoom into a specific area
ax = plt.gca()
ax.set_xlim(290000, 303000)
ax.set_ylim(4095000, 4110000)

plt.show()

For reference, here is a zoomed-in RGB image of the SOAP site along with the Creek Fire boundary, around our tiles of interest (unburned in green, burned in orange).

![png](./notebook_graphics/soap_rois.png)

Lastly, we can take a quick look at the CHMs for these two tiles, plotting maps and histograms to get an initial sense for the structural composition of the forest. See https://www.neonscience.org/resources/learning-hub/tutorials/classify-chm-py for more details. The `rasterio` package, imported as `rio` makes reading in geotiff rasters very simple.

In [ ]:
burned_chm_tile = '..\\..\\..\\data\\SOAP\\NEON\\DP3.30015.001\\neon-aop-products\\2023\\FullSite\\D17\\2023_SOAP_7\\l3\\DiscreteLidar\\CanopyHeightModelGtif\\NEON_D17_SOAP_DP3_298000_4100000_CHM.tif'
unburned_chm_tile = '..\\..\\..\\data\\SOAP\\NEON\\DP3.30015.001\\neon-aop-products\\2023\\FullSite\\D17\\2023_SOAP_7\\l3\\DiscreteLidar\\CanopyHeightModelGtif\\NEON_D17_SOAP_DP3_298000_4101000_CHM.tif'
burned_chm_dataset = rio.open(burned_chm_tile)
unburned_chm_dataset = rio.open(unburned_chm_tile)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))
show(burned_chm_dataset, ax=ax1);
ax1.ticklabel_format(style='plain', axis='y') 
ax1.set_title('CHM of Burned Tile')

show_hist(burned_chm_dataset, bins=50, histtype='stepfilled',
          lw=0.0, stacked=False, alpha=0.3, ax=ax2);
ax2.set_xlabel("Canopy Height (meters)");
ax2.get_legend().remove()
ax2.set_title('CHM Histogram')

plt.show();

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,5))
show(unburned_chm_dataset, ax=ax1);
ax1.ticklabel_format(style='plain', axis='y') 
ax1.set_title('CHM of Unburned Tile')

show_hist(unburned_chm_dataset, bins=50, histtype='stepfilled',
          lw=0.0, stacked=False, alpha=0.3, ax=ax2);
ax2.set_xlabel("Canopy Height (meters)");
ax2.get_legend().remove()
ax2.set_title('CHM Histogram')

plt.show();

What can you tell from these maps and histograms?

## Contact Info:  

**National Ecological Observatory Network (NEON)**<sup>2</sup>

Website: <https://www.neonscience.org/>   
Contact: <https://www.neonscience.org/about/contact-us>   
Date last modified: 08-27-2024 

<sup>2</sup>NEON is a major facilitiy fully funded by the National Science Foundation and operated by Battelle.